<a href="https://colab.research.google.com/github/cassiodatacyber/Ci-ncia-de-Dados/blob/main/Projeto_Elabora%C3%A7%C3%A3o_de_uma_Rede_Neural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import os

# Adiciona a pasta principal ao sys.path para podermos importar modulos de myutils
main_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if main_path not in sys.path:
    sys.path.append(main_path)

# silenciar avisos do TF
os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_VLOG_LEVEL'] = '3'


import tensorflow as tf
from tensorflow import keras as K
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from myutils import imtools, models, eval
import importlib
from absl import logging

# Define o nível de logging para silenciar os avisos do TF
tf.get_logger().setLevel('ERROR')
logging.set_verbosity(logging.ERROR)
  # Tenta silenciar logs mais verbosos

# Toy set - XOR Gate

In [ ]:
# metadados
N = 200
MEAN = 10
STD = 3
OFFSET = 10

# a distribuição XOR contém 4 regiões.
# Neste exemplo, cada região é um centro gaussiano 2D
centers = [
    [MEAN, MEAN],
    [-MEAN, -MEAN],
    [MEAN, -MEAN],
    [-MEAN, MEAN],
]

# futuros tensores X e y, respectivamente
data = []
targets = []

# para cada centro, crie uma gaussiana 2D com um 1/4 de N total
for center in centers:
    x = np.random.normal(loc=center, scale=STD, size=(N//4, 2))
    target = 1 if np.mean(center) == 0 else 0 # lógica XOR
    data.append(x + OFFSET) # adicionar offset para criar assimetria nas escalas
    targets.append(np.ones((N//4,)) * target) # cada y pode ser 1 ou 0 de acordo com 'target'

# transformando as listas em tensores numpy
X = np.concatenate(data, axis=0)
y = np.concatenate(targets, axis=0)

# visualizando scatter dos dados
plt.figure(figsize=(10, 6))
plt.scatter(X[y==0, 0], X[y==0, 1], s=50, marker='o', c='blue', label='1') # utilize : para representar 'todos os pontos do eixo específico' | [:, 0] => 'todos os pontos eixo 0, apenas o primeiro ponto do eixo 1'
plt.scatter(X[y==1, 0], X[y==1, 1], s=50, marker='d', c='red', label='0')
plt.title('Lógica XOR')
plt.xlabel('B')
plt.ylabel('A')
plt.legend()
plt.show()

# MLP1 - Baseline

In [ ]:
importlib.reload(models)

model = models.build_model_mlp_1(input_shape=X.shape[1:])

model.summary()
K.utils.plot_model(model, show_shapes=True, show_layer_names=True)

Dica: Explore o site do keras para conhecer diferentes funções de perda, otimizadores, e métricas

- otimizadores: [keras optimizers](https://keras.io/api/optimizers/)
- losses: [keras losses](https://keras.io/api/losses/)
- métricas: [keras metrics](https://keras.io/api/metrics/)


exemplo de uso:
```python
model.compile(
    optimizer=K.optimizers.SGD(learning_rate=1),
    loss=K.losses.MeanAbsoluteError(),
    metrics=[K.metrics.Accuracy()]
)
```

Se possível, explore outros!


In [ ]:
model.compile(
    optimizer=...,
    loss=...,
    metrics=[...]
)

hist = model.fit(
    X, y,
    epochs=...,
    batch_size=...,
    shuffle=True
)

history_df = pd.DataFrame(hist.history)
history_df.head()

In [ ]:
# visualizando loss e acc (não precisa mexer)
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Plot loss
axes[0].plot(history_df.index, history_df['loss'], label='Loss', color='blue')
axes[0].set_title('Model Loss')
axes[0].set_ylabel('Loss')
axes[0].legend()

# Plot accuracy
axes[1].plot(history_df.index, history_df['accuracy'], label='Accuracy', color='green')
axes[1].set_title('Model Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# extraindo predição de todos X
y_pred = model.predict(X)

In [ ]:
# comparando predições vs ground truth
eval.scatter_binary(X, y, y_pred)

# MLP2 - Exploração do Problema

- Explore novas configurações de modelo livremente.
- Explore reconfigurar as features A e B livremente.
- Explore diferentes taxas de aprendizado.
- Explore diferentes batch sizes.
- Explore diferentes números de épocas.

*Objetivos:*
- Top1: Alcançar o melhor resultado possível.
- Top2: Alcançar o menor modelo possível.
- Top3: Alcançar o menor número de épocas possível.

In [ ]:
importlib.reload(models)

model = ...

model.summary()
K.utils.plot_model(model, show_shapes=True, show_layer_names=True)

In [ ]:
model.compile(
    optimizer=...,
    loss=...,
    metrics=[...]
)

hist = model.fit(
    X, y,
    epochs=...,
    batch_size=...,
    shuffle=True
)

history_df = pd.DataFrame(hist.history)
history_df.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Plot loss
axes[0].plot(history_df.index, history_df['loss'], label='Loss', color='blue')
axes[0].set_title('Model Loss')
axes[0].set_ylabel('Loss')
axes[0].legend()

# Plot accuracy
axes[1].plot(history_df.index, history_df['accuracy'], label='Accuracy', color='green')
axes[1].set_title('Model Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
y_pred = model.predict(X)

In [ ]:
eval.scatter_binary(X, y, y_pred)